In [1]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# ---- Load current model predictions ----
preds_full = pd.read_pickle("PickleFiles/Full PPR Rankings.pkl")
preds_half = pd.read_pickle("PickleFiles/Half PPR Rankings.pkl")
preds_std  = pd.read_pickle("PickleFiles/Non PPR Rankings.pkl")

# ---- Fetch 2024 actual stats ----
print("Fetching 2024 seasonal data from nfl_data_py...")
seasonal_2024 = nfl.import_seasonal_data([2024])
rosters_2024  = nfl.import_seasonal_rosters([2024])
rosters_2024  = rosters_2024[rosters_2024['position'].isin(['QB','RB','WR','TE'])].copy()
rosters_2024  = rosters_2024[['player_id','player_name','position']].drop_duplicates('player_id')

# Merge names + position into seasonal stats
stats = seasonal_2024.merge(rosters_2024, on='player_id', how='inner')
stats = stats[stats['position'].isin(['QB','RB','WR','TE'])].copy()

# fantasy_points in nfl_data_py = Full PPR (1 pt/reception)
stats['actual_ppr']  = stats['fantasy_points'] / stats['games']
stats['actual_half'] = (stats['fantasy_points'] - stats['receptions'] * 0.5) / stats['games']
stats['actual_std']  = (stats['fantasy_points'] - stats['receptions']) / stats['games']

# Keep players with >= 8 games (avoids injury-shortened samples)
stats = stats[stats['games'] >= 8].copy()
stats = stats.rename(columns={'player_name': 'Name'})

print(f"2024 actuals: {len(stats)} players (>= 8 games played)")
print(f"Full PPR predictions: {len(preds_full)} players")

/Users/kmaran3/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Fetching 2024 seasonal data from nfl_data_py...


URLError: <urlopen error [Errno 8] nodename nor servname provided, or not known>

In [ ]:
def evaluate(preds_df, actuals_df, actual_col, scoring_name):
    merged = preds_df[['Name','Position','Final PPG']].merge(
        actuals_df[['Name', actual_col, 'position']], 
        on='Name', how='inner'
    )
    if len(merged) < 10:
        print(f"{scoring_name}: only {len(merged)} name matches — check name format")
        return None
    
    mae  = mean_absolute_error(merged[actual_col], merged['Final PPG'])
    rmse = np.sqrt(((merged[actual_col] - merged['Final PPG'])**2).mean())
    corr, pval = pearsonr(merged['Final PPG'], merged[actual_col])
    
    merged['Error']     = merged['Final PPG'] - merged[actual_col]
    merged['Abs_Error'] = merged['Error'].abs()
    
    print(f"
{'='*55}")
    print(f" {scoring_name}")
    print(f"{'='*55}")
    print(f"  Matched players: {len(merged)}")
    print(f"  MAE:             {mae:.2f} PPG")
    print(f"  RMSE:            {rmse:.2f} PPG")
    print(f"  Correlation:     {corr:.3f}  (p={pval:.4f})")
    
    print(f"
  By position (players with >= 8 games):")
    for pos in ['QB','RB','WR','TE']:
        sub = merged[merged['position'] == pos]
        if len(sub) >= 5:
            pm  = mean_absolute_error(sub[actual_col], sub['Final PPG'])
            pc, _ = pearsonr(sub['Final PPG'], sub[actual_col])
            print(f"    {pos}: n={len(sub):3d} | MAE={pm:.2f} | r={pc:.3f}")
    
    print(f"
  10 worst predictions (highest absolute error):")
    worst = merged.nlargest(10, 'Abs_Error')[['Name','position','Final PPG',actual_col,'Error']]
    worst.columns = ['Name','Pos','Pred','Actual','Error']
    print(worst.to_string(index=False))
    
    return merged

eval_full = evaluate(preds_full, stats, 'actual_ppr',  'FULL PPR — Current Model')
eval_half = evaluate(preds_half, stats, 'actual_half', 'HALF PPR — Current Model')
eval_std  = evaluate(preds_std,  stats, 'actual_std',  'STANDARD — Current Model')

SyntaxError: unterminated string literal (detected at line 17) (4287841897.py, line 17)

In [ ]:
print("
" + "="*55)
print(" BASELINE SUMMARY — Current Model vs 2024 Actuals")
print("="*55)
print(f"{'Format':<15} {'Matched':>8} {'MAE (PPG)':>12} {'Correlation':>13}")
print("-"*55)
for name, ev, col in [('Full PPR', eval_full, 'actual_ppr'),
                       ('Half PPR', eval_half, 'actual_half'),
                       ('Standard', eval_std,  'actual_std')]:
    if ev is not None and len(ev) > 5:
        m = mean_absolute_error(ev[col], ev['Final PPG'])
        c, _ = pearsonr(ev['Final PPG'], ev[col])
        print(f"{name:<15} {len(ev):>8} {m:>12.2f} {c:>13.3f}")
    else:
        print(f"{name:<15}    N/A")
print()
print("Run ImprovedMLModel.ipynb to retrain and compare.")